In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

massey = pd.read_csv(RAW_DIR / "MMasseyOrdinals.csv")

print("Shape:", massey.shape)

display(massey.head())

print("\nColumns:")
print(massey.columns.tolist())

Shape: (5865001, 5)


,Season,RankingDayNum,SystemName,TeamID,OrdinalRank
0,2003,35,SEL,1102,159
1,2003,35,SEL,1103,229
2,2003,35,SEL,1104,12
3,2003,35,SEL,1105,314
4,2003,35,SEL,1106,260



Columns:
['Season', 'RankingDayNum', 'SystemName', 'TeamID', 'OrdinalRank']


In [2]:
print("Season range:")
print(massey["Season"].min(), "-", massey["Season"].max())

print("\nNumber of ranking systems:")
print(massey["SystemName"].nunique())

print("\nNumber of teams:")
print(massey["TeamID"].nunique())

print("\nMissing values:")
print(massey.isna().sum())

print("\nDuplicate exact rows:")
print(massey.duplicated().sum())

Season range:
2003 - 2026

Number of ranking systems:
197

Number of teams:
372

Missing values:
Season           0
RankingDayNum    0
SystemName       0
TeamID           0
OrdinalRank      0
dtype: int64

Duplicate exact rows:
0


In [3]:
system_counts = (
    massey.groupby("SystemName")
    .agg(
        Seasons=("Season", "nunique"),
        FirstSeason=("Season", "min"),
        LastSeason=("Season", "max"),
        Rows=("OrdinalRank", "size")
    )
    .sort_values(
        ["Seasons", "Rows"],
        ascending=False
    )
)

display(system_counts.head(30))

,Seasons,FirstSeason,LastSeason,Rows
SystemName,,,,
MOR,24,2003,2026,150475
POM,24,2003,2026,147984
WLK,24,2003,2026,120647
DOL,24,2003,2026,117809
COL,24,2003,2026,111545
USA,24,2003,2026,10916
AP,24,2003,2026,10905
BIH,23,2003,2026,104330
WOL,23,2003,2026,94224


In [4]:
ranking_days = (
    massey.groupby("Season")["RankingDayNum"]
    .agg(["min", "max", "nunique"])
)

display(ranking_days.tail(15))

print("Most common latest ranking days:")

display(
    ranking_days["max"]
    .value_counts()
    .sort_index()
)

,min,max,nunique
Season,,,
2012,7,133,66
2013,6,133,53
2014,9,133,19
2015,16,133,18
2016,16,133,18
2017,16,133,18
2018,16,133,18
2019,9,133,19
2020,9,128,18


Most common latest ranking days:


max
128     1
133    23
Name: count, dtype: int64

In [5]:
pre_tournament = massey[
    massey["RankingDayNum"].between(128, 133)
]

pre_tournament_systems = (
    pre_tournament.groupby("SystemName")
    .agg(
        Seasons=("Season", "nunique"),
        FirstSeason=("Season", "min"),
        LastSeason=("Season", "max"),
        Teams=("TeamID", "nunique"),
        Rows=("OrdinalRank", "size")
    )
    .sort_values(
        ["Seasons", "Rows"],
        ascending=False
    )
)

display(pre_tournament_systems.head(30))

,Seasons,FirstSeason,LastSeason,Teams,Rows
SystemName,,,,,
MOR,24,2003,2026,372,13962
POM,24,2003,2026,372,13959
WLK,24,2003,2026,372,13594
DOL,24,2003,2026,372,13593
COL,24,2003,2026,371,13592
USA,24,2003,2026,107,977
AP,24,2003,2026,106,976
BIH,23,2003,2026,372,12255
RTH,22,2003,2026,372,10829


In [6]:
PRE_TOURNAMENT_DAY = 133

pre_tourney = (
    massey[
        massey["RankingDayNum"] <= PRE_TOURNAMENT_DAY
    ]
    .sort_values("RankingDayNum")
    .groupby(
        ["Season", "SystemName", "TeamID"],
        as_index=False
    )
    .tail(1)
)

print("Pre-tournament rows:", len(pre_tourney))

display(pre_tourney.head())

Pre-tournament rows: 458694


,Season,RankingDayNum,SystemName,TeamID,OrdinalRank
679242,2008,1,AP,1246,20
680282,2008,1,USA,1246,22
1488489,2012,7,AP,1417,17
1737317,2013,7,D1A,1365,43
1737288,2013,7,D1A,1172,46


In [7]:
FULL_COVERAGE_SYSTEMS = [
    "MOR",
    "POM",
    "WLK",
    "DOL",
    "COL"
]

snapshot_coverage = (
    pre_tourney[
        pre_tourney["SystemName"].isin(FULL_COVERAGE_SYSTEMS)
    ]
    .groupby(["SystemName", "Season"])
    .agg(
        Teams=("TeamID", "nunique"),
        LatestDay=("RankingDayNum", "max"),
        EarliestUsedDay=("RankingDayNum", "min")
    )
    .reset_index()
)

display(snapshot_coverage)

,SystemName,Season,Teams,LatestDay,EarliestUsedDay
0,COL,2003,327,133,133
1,COL,2004,326,133,133
2,COL,2005,330,133,133
3,COL,2006,334,133,133
4,COL,2007,336,133,133
...,...,...,...,...,...
115,WLK,2022,358,133,133
116,WLK,2023,363,133,133
117,WLK,2024,362,133,133
118,WLK,2025,364,133,133


In [8]:
system_snapshot_summary = (
    snapshot_coverage
    .groupby("SystemName")
    .agg(
        Seasons=("Season", "nunique"),
        MinTeams=("Teams", "min"),
        MaxTeams=("Teams", "max"),
        MedianTeams=("Teams", "median")
    )
    .sort_values("MinTeams", ascending=False)
)

display(system_snapshot_summary)

,Seasons,MinTeams,MaxTeams,MedianTeams
SystemName,,,,
COL,24,326,364,349.5
DOL,24,326,365,349.0
MOR,24,326,365,351.0
POM,24,326,365,351.0
WLK,24,326,365,349.0


In [10]:
men_processed = pd.read_csv(
    PROCESSED_DIR / "men_processed.csv"
)

tournament_teams = pd.concat([
    men_processed[["Season", "Team1"]]
        .rename(columns={"Team1": "TeamID"}),

    men_processed[["Season", "Team2"]]
        .rename(columns={"Team2": "TeamID"})
]).drop_duplicates()

coverage_checks = []

for system in FULL_COVERAGE_SYSTEMS:

    system_ranks = (
        pre_tourney[
            pre_tourney["SystemName"] == system
        ][["Season", "TeamID", "OrdinalRank"]]
    )

    checked = tournament_teams.merge(
        system_ranks,
        on=["Season", "TeamID"],
        how="left"
    )

    coverage_checks.append({
        "SystemName": system,
        "TournamentTeams": len(checked),
        "MissingRanks": checked["OrdinalRank"].isna().sum(),
        "CoveragePct": (
            checked["OrdinalRank"].notna().mean() * 100
        )
    })

coverage_checks = pd.DataFrame(coverage_checks)

display(coverage_checks)

,SystemName,TournamentTeams,MissingRanks,CoveragePct
0,MOR,1471,0,100.0
1,POM,1471,0,100.0
2,WLK,1471,0,100.0
3,DOL,1471,0,100.0
4,COL,1471,0,100.0


In [11]:
consensus_source = pre_tourney[
    pre_tourney["SystemName"].isin(FULL_COVERAGE_SYSTEMS)
].copy()

consensus_rank = (
    consensus_source
    .groupby(["Season", "TeamID"])
    .agg(
        ConsensusRank=("OrdinalRank", "mean"),
        SystemsUsed=("SystemName", "nunique")
    )
    .reset_index()
)

print("Shape:", consensus_rank.shape)

display(consensus_rank.head())

print("\nSystems used per team-season:")
print(consensus_rank["SystemsUsed"].value_counts().sort_index())

Shape: (8355, 4)


,Season,TeamID,ConsensusRank,SystemsUsed
0,2003,1102,158.8,5
1,2003,1103,164.0,5
2,2003,1104,35.4,5
3,2003,1105,310.2,5
4,2003,1106,266.6,5



Systems used per team-season:
SystemsUsed
1       3
2       9
3       2
4       3
5    8338
Name: count, dtype: int64


In [12]:
consensus_check = tournament_teams.merge(
    consensus_rank,
    on=["Season", "TeamID"],
    how="left"
)

print(
    "Tournament team-seasons:",
    len(consensus_check)
)

print(
    "Missing consensus ranks:",
    consensus_check["ConsensusRank"].isna().sum()
)

print("\nNumber of systems used:")
print(
    consensus_check["SystemsUsed"]
    .value_counts()
    .sort_index()
)

Tournament team-seasons: 1471
Missing consensus ranks: 0

Number of systems used:
SystemsUsed
5    1471
Name: count, dtype: int64


In [13]:
men_massey = men_processed.copy()

# Team1 consensus rank
men_massey = men_massey.merge(
    consensus_rank[
        ["Season", "TeamID", "ConsensusRank"]
    ].rename(columns={
        "TeamID": "Team1",
        "ConsensusRank": "Team1MasseyRank"
    }),
    on=["Season", "Team1"],
    how="left"
)

# Team2 consensus rank
men_massey = men_massey.merge(
    consensus_rank[
        ["Season", "TeamID", "ConsensusRank"]
    ].rename(columns={
        "TeamID": "Team2",
        "ConsensusRank": "Team2MasseyRank"
    }),
    on=["Season", "Team2"],
    how="left"
)

men_massey["MasseyRankDiff"] = (
    men_massey["Team1MasseyRank"]
    - men_massey["Team2MasseyRank"]
)

display(men_massey.head())

,Season,Team1,Team2,SeedDiff,AvgScoringMarginDiff,WinRateDiff,EFGDifferentialDiff,Target,Team1MasseyRank,Team2MasseyRank,MasseyRankDiff
0,2003,1411,1421,0,9.208046,0.151724,0.045943,0,250.4,251.6,-1.2
1,2003,1112,1436,-15,10.309113,0.237685,0.044359,1,2.6,156.8,-154.2
2,2003,1113,1272,3,-1.896552,-0.172414,-0.024137,1,34.8,20.8,14.0
3,2003,1141,1166,5,-8.805643,-0.085684,-0.020792,1,40.8,23.0,17.8
4,2003,1143,1301,-1,0.324138,0.124138,-0.007780,1,33.8,47.0,-13.2


In [14]:
print("Rows:", len(men_massey))

print(
    "Missing Team1 Massey:",
    men_massey["Team1MasseyRank"].isna().sum()
)

print(
    "Missing Team2 Massey:",
    men_massey["Team2MasseyRank"].isna().sum()
)

print(
    "Missing MasseyRankDiff:",
    men_massey["MasseyRankDiff"].isna().sum()
)

print("\nMasseyRankDiff summary:")

display(
    men_massey["MasseyRankDiff"].describe()
)

Rows: 1449
Missing Team1 Massey: 0
Missing Team2 Massey: 0
Missing MasseyRankDiff: 0

MasseyRankDiff summary:


count    1449.00000
mean        0.58882
std        66.80761
min      -275.80000
25%       -24.80000
50%        -1.00000
75%        26.00000
max       288.80000
Name: MasseyRankDiff, dtype: float64

In [15]:
MASSEY_OUTPUT = PROCESSED_DIR / "men_massey.csv"

men_massey.to_csv(
    MASSEY_OUTPUT,
    index=False
)

print("Saved:", MASSEY_OUTPUT)
print("Shape:", men_massey.shape)

Saved: ../data/processed/men_massey.csv
Shape: (1449, 11)
